Now that I understood what are the things inside the data, I managed to create three data frames. 

In the challenge they said **The best solutions will likely come from clever feature engineering that can extract the most predictive signals from both the initial matchup and the flow of the early battle.**

So I tried to make two things:

1. A dataframe of initial matchup - ie, some of the details of all 6 pokemon of p1 and lead pokemon of p2(we can extract more info by using the same code.):

   - Along with that I created another dataframe of P1 lead pokemon vs P2 lead pokemon, to check whether the first pokemon of P1 have any advantage disadvantage.
   - We can also check the item matchup between P1 lead vs P2 lead pokemon.
   - Also I want to discuss; rather than these raw data, we could have something like total stat strength of P1, to get an idea whether the pokemon team he put in each battle is whether aggressive or defensive. 
    
    
    
3. Battle flow : This is the turn by turn snapshot of what is happening, we can check if the hp is decreasing or increasing, whether P2 put negative effects on P1, final state of the pokemon at the 30th turn etc.

So you can comment edit this notebook! put comments or what you think by making a markdown cell.

In [ ]:
import json
import pandas as pd
import numpy as np


# ==========================================
# 1. LOAD THE TRAINING DATA
# ==========================================
train_file_path = 'data/train.jsonl'
train_data = []

with open(train_file_path, 'r') as f:
    for line in f:
        train_data.append(json.loads(line))

print(f"Successfully loaded {len(train_data)} training battles.")



# ==========================================
# 2. STATIC MATCH UP
# ==========================================

initial_matchup = []

lead_matchup = []

all_turns = []

player_win_counter = 0

for battle in train_data:

    player_won = battle.get('player_won', None)




for battle in train_data:
    
    battle_id = battle.get('battle_id')

    player_won = battle.get('player_won', None)
    
    # counting the number of times the player 1 - won the battle
    if player_won == 1:
        player_win_counter+=1

        
    # ==========================================
    # 2. INITIAL MATCH UP 
    # ==========================================

    # Player 1 details
    p1_team = battle.get('p1_team_details',[])

    # Player 1 Lead details
    p1_lead = p1_team[0] if len(p1_team) > 0 else {}
    p1_lead_name = p1_lead.get('name', 'Unknown')
    p1_lead_type = p1_lead.get('types', [])
    p1_lead_item = p1_lead.get('item', 'NaN')

    # This is all the Player 1 pokemon details - you can extract stats, level etc with same as below 
    # then with the Dataframe we can take mean, max, difference etc.
    p1_names = [pok.get('name', 'Unknown') for pok in p1_team]
    p1_types = [pok.get('types', []) for pok in p1_team]
    p1_items = [pok.get('item', 'NaN') for pok in p1_team]

    # Player 2 lead details - No item in the player 2
    p2_lead = battle.get('p2_lead_details', {}) or {}
    p2_lead_name = p2_lead.get('name', 'Unknown')
    p2_lead_type = p2_lead.get('types', [])


    initial_matchup.append({
        'battle_id': battle_id,
        'p1_names':p1_names,
        'p1_types':p1_types,
        'p1_items':p1_items,
        'p2_lead_name': p2_lead_name,
        'p2_lead_type': p2_lead_type,
        'player_won': player_won


        
    })
   

    lead_matchup.append({
        'battle_id': battle_id,
        'p1_lead_name': p1_lead_name,
        'p1_lead_type': p1_lead_type,
        'p1_lead_item': p1_lead_item,
        'p2_lead_name': p2_lead_name,
        'p2_lead_type': p2_lead_type,
        'player_won': player_won
    })



    # ===================================================
    # 3. THIS IS THE FLOW OF THE EARLY BATTLE - 30 TURN
    # ===================================================

    timeline = battle.get('battle_timeline', [])

    
    for turn_data in timeline:
        turn_number = turn_data.get('turn')

        #pokemon state details
        p1_state = turn_data.get('p1_pokemon_state', {})
        p1_name = p1_state.get('name', 'Unknown')
        p1_hp = p1_state.get('hp_pct', 0.0)
        p1_status = p1_state.get('status', 'Unknown')
        p1_effects = p1_state.get('effects',[])

        #move details
        p1_move = turn_data.get('p1_move_details',{}) or {}
        p1_type = p1_move.get('type', 'NaN')
        p1_category = p1_move.get('category', 'NaN')
        
        

        
        p2_state = turn_data.get('p2_pokemon_state', {})
        p2_name = p2_state.get('name', 'Unknown')
        p2_hp = p2_state.get('hp_pct', 0.0)
        p2_status = p2_state.get('status', 'Unknown')
        p2_effects = p2_state.get('effects',[])

        p2_move = turn_data.get('p2_move_details',{}) or {}
        p2_type = p2_move.get('type', 'NaN')
        p2_category = p2_move.get('category', 'NaN')

        
        
        # including the battle outcome
        all_turns.append({
            'battle_id': battle_id,
            'player_won': player_won, 
            'turn': turn_number,
            'p1_active_pokemon': p1_name,
            'p1_hp_pct': p1_hp,
            'p1_status': p1_status,
            'p1_effects': p1_effects,
            'p1_move_type': p1_type,
            'p1_move_category': p1_category,
            'p2_active_pokemon': p2_name,
            'p2_hp_pct': p2_hp,
            'p2_status': p2_status,
            'p2_effects': p2_effects,
            'p2_move_type': p2_type,
            'p2_move_category': p2_category,
            
        })

# Convert to three DataFrames - P1 lead vs P2 Lead, intital pokemon details and battle flow
df_lead_matchup = pd.DataFrame(lead_matchup)
df_initial_matchup = pd.DataFrame(initial_matchup)
df_battleflow = pd.DataFrame(all_turns)

print("\n","="*20, "P1 lead vs P2 lead", "="*20,"\n")
display(df_lead_matchup.head(5))

print("\n","="*20, "P1 all pokemon vs P2 lead", "="*20,"\n")
display(df_initial_matchup.head(5))

print("\n","="*20, "Turn by Turn Battle", "="*20,"\n")
display(df_battleflow.head(30))



#display(df_snapshots.iloc[30:60])

# calculating the win percentage by normal method
player_winrate = player_win_counter/len(train_data)
print(f"Player 1 Win Rate :{player_winrate * 100} % of the time")

# calculating the win percentage by pandas oneliner
print(f"Player 1 Win Rate: {df_initial_matchup['player_won'].mean() * 100:.2f}%")


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/train.jsonl'

In [ ]:
# Using pandas to load the train and test
'''
import json
import pandas as pd
from pathlib import Path

train_file_path = '/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/train.jsonl'
test_file_path = '/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/test.jsonl'


df_train = pd.read_json(train_file_path, lines=True)

df_test = pd.read_json(test_file_path, lines = True)


display(df_train.head(5))
display(df_test.head(5))

'''



In [ ]:
'''
# p1_team_details is a list of dictionaries, so I use explode() function 
# to iterate over the list of dict and make rows
df_explode_p1team = df_train.explode('p1_team_details').reset_index(drop=True)

# expected output is 6 pokemon per battle for 10000 battle = 60000
df_explode_p1team.info() 


print("-"*100)

# The new rows contains dictionaries and I make coloumns for each dict keys
df_flat_p1team = pd.json_normalize(df_explode_p1team['p1_team_details'])
df_flat_p1team.info()
'''

In [ ]:
'''
# p2_lead_details is a single dictionary, so no need to explode

df_flat_p2team = pd.json_normalize(df_train['p2_lead_details'])

# expected output is 1 lead pokemon per battle for 10000 battle = 10000
df_flat_p2team.info()
'''

In [ ]:
'''
df_explode_battle_timeline = df_train.explode('battle_timeline').reset_index(drop=True)

# expected output is 30 turn per battle_timeline with a total 10000 battle = 300000
df_explode_battle_timeline.info()

print("-"*100)

df_flat_battle_timeline = pd.json_normalize(df_explode_battle_timeline['battle_timeline'])

df_flat_battle_timeline.info()

'''

**With this I identified some variables type as objects and I will try to further inspect 
that and try to print out their contents. 
My aim is to make categorical values that give us some more info about the battles and 
there by improving our features, training and prediction**

In [ ]:
'''
# Loop through each column in the dataframe
for column_name in df_flat_p1team.columns:
    
    # Check if the data type is string (object)
    if df_flat_p1team[column_name].dtype == 'object':
        
        print(f"Looking at column: {column_name}")
        
        # Create a variable for the values
        column_data = df_flat_p1team[column_name].astype(str)
        
        # Count the values
        top_values = column_data.value_counts()
        print(top_values.tail())
        print("-" * 20)

'''

For player 1 we can list out all the pokemons, their types and items used for the battle

In [ ]:
'''
print(f"Name of the Poekmons = \n{df_flat_p1team['name'].unique()}\n")

print(f"Pokemon Types = \n{df_flat_p1team['types'].explode().unique()}\n")

print(f"Items =\n {df_flat_p1team['item'].unique()}")

'''

For Player 2 we have only one information, which is the first pokemon it is leading with and it's type

In [ ]:
'''
# Loop through each column in the dataframe
for column_name in df_flat_p2team.columns:
    
    # Check if the data type is string (object)
    if df_flat_p2team[column_name].dtype == 'object':
        
        print(f"Looking at column: {column_name}")
        
        # Create a variable for the values
        column_data = df_flat_p2team[column_name].astype(str)
        
        # Count the values
        top_values = column_data.value_counts()
        print(top_values.tail())
        print("-" * 20)
'''

In [ ]:
'''
print(f"Name of the Poekmons = \n{df_flat_p2team['name'].unique()}\n")

print(f"Pokemon Types = \n{df_flat_p2team['types'].explode().unique()}\n")
'''

Now we go through the Battle Timeline and extract some information which is relevant such as the status of pokemons during the battle like paralysis, fire, freeze, poison. 

??may be we can use them to give weightage to the chance of winning or loosing

In [ ]:
'''

# Loop through each column in the dataframe
for column_name in df_flat_battle_timeline.columns:
    
    # Check if the data type is string (object)
    if df_flat_battle_timeline[column_name].dtype == 'object':
        
        print(f"Looking at column: {column_name}")
        
        # Create a variable for the values
        column_data = df_flat_battle_timeline[column_name].astype(str)
        
        # Count the values
        top_values = column_data.value_counts()
        print(top_values.tail())
        print("-" * 20)

'''

Now we list all the status pokemon gone through in the battle timeline

In [ ]:
'''

# Select both P1 and P2 Pokemon Status, then explode them to break the lists apart
p1_status = df_flat_battle_timeline['p1_pokemon_state.status'].explode()
p2_status = df_flat_battle_timeline['p2_pokemon_state.status'].explode()

# Combine and find the unique values
all_status = pd.concat([p1_status, p2_status]).unique()

print(all_status)

'''

Another important one - which is the sepcial effects used by the pokemons

In [ ]:
'''

# Select both P1 and P2 effects, then explode them to break the lists apart
p1_effects = df_flat_battle_timeline['p1_pokemon_state.effects'].explode()
p2_effects = df_flat_battle_timeline['p2_pokemon_state.effects'].explode()

# Combine and find the unique values
all_effects = pd.concat([p1_effects, p2_effects]).unique()

print(all_effects)
#type(all_effects)

'''

This tells us the elemental move. 

Why important? example: water VS fire, fire takes damage

In [ ]:
'''

# Select both P1 and P2 move types, then explode them to break the lists apart
p1_move_type = df_flat_battle_timeline['p1_move_details.type'].explode()
p2_move_type = df_flat_battle_timeline['p2_move_details.type'].explode()

# Combine and find the unique values
all_move_type = pd.concat([p1_move_type, p2_move_type]).unique()

print(all_move_type)

'''


This tells us the move class:

- physical -> attack/defense
- special -> special attack/defense
- status -> (I don't know)
- nan -> (I suspect not doing anything/lost/switch pokemon)

In [ ]:
'''


# Select both P1 and P2 move categories, then explode them to break the lists apart
p1_move_category = df_flat_battle_timeline['p1_move_details.category'].explode()
p2_move_category = df_flat_battle_timeline['p2_move_details.category'].explode()

# Combine and find the unique values
all_move_category = pd.concat([p1_move_category, p2_move_category]).unique()

print(all_move_category)

'''

In [ ]:
import json
import pandas as pd
from pathlib import Path

train_file_path = '/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/train.jsonl'
test_file_path = '/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/test.jsonl'
sample_submission_path = '/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/sample_submission.csv'
train_data = []

# Read the file line by line
print(f"Loading data from '{train_file_path}'...")
try:
    with open(train_file_path, 'r') as f:
        for line in f:
            # json.loads() parses one line (one JSON object) into a Python dictionary
            train_data.append(json.loads(line))

    print(f"Successfully loaded {len(train_data)} battles.")

    # Let's inspect the first battle to see its structure
    print("\n--- Structure of the first train battle: ---")
    if train_data:
        first_battle = train_data[0]
        
        # To keep the output clean, we can create a copy and truncate the timeline
        battle_for_display = first_battle.copy()
        battle_for_display['battle_timeline'] = battle_for_display.get('battle_timeline', [])[:2] # Show first 2 turns
        
        # Use json.dumps for pretty-printing the dictionary
        print(json.dumps(battle_for_display, indent=4))
        if len(first_battle.get('battle_timeline', [])) > 3:
            print("    ...")
            print("    (battle_timeline has been truncated for display)")


except FileNotFoundError:
    print(f"ERROR: Could not find the training file at '{train_file_path}'.")
    print("Please make sure you have added the competition data to this notebook.")

### 2. Basic Feature Engineering

A successful model will likely require creating many complex features. For this starter notebook, however, we will create a very simple feature set based **only on the initial team stats**. This will be enough to train a model and generate a submission file.

It's up to you to engineer more powerful features!

In [ ]:
from tqdm.notebook import tqdm
import numpy as np

def create_simple_features(data):
    """
    A very basic feature extraction function.
    It only uses the aggregated base stats of the player's team and opponent's lead.
    """
    feature_list = []
    for battle in tqdm(data, desc="Extracting features"):
        features = {}
        
        # --- Player 1 Team Features ---
        p1_team = battle.get('p1_team_details', [])
        if p1_team:
            features['p1_mean_hp'] = np.mean([p.get('base_hp', 0) for p in p1_team])
            features['p1_mean_spe'] = np.mean([p.get('base_spe', 0) for p in p1_team])
            features['p1_mean_atk'] = np.mean([p.get('base_atk', 0) for p in p1_team])
            features['p1_mean_def'] = np.mean([p.get('base_def', 0) for p in p1_team])

        # --- Player 2 Lead Features ---
        p2_lead = battle.get('p2_lead_details')
        if p2_lead:
            features['p2_lead_hp'] = p2_lead.get('base_hp', 0)
            features['p2_lead_spe'] = p2_lead.get('base_spe', 0)

        # We also need the ID and the target variable (if it exists)
        features['battle_id'] = battle.get('battle_id')
        if 'player_won' in battle:
            features['player_won'] = int(battle['player_won'])
            
        feature_list.append(features)
        
    return pd.DataFrame(feature_list).fillna(0)

# Create feature DataFrames for both training and test sets
print("Processing training data...")
train_df = create_simple_features(train_data)

print("\nProcessing test data...")
test_data = []
with open(test_file_path, 'r') as f:
    for line in f:
        test_data.append(json.loads(line))
test_df = create_simple_features(test_data)

print("\nTraining features preview:")
display(train_df.head())

### 3. Training a Baseline Model

Now that we have some features, let's train a simple `LogisticRegression` model. This will give us a starting point for our predictions.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Define our features (X) and target (y)
features = [col for col in train_df.columns if col not in ['battle_id', 'player_won']]
X_train = train_df[features]
y_train = train_df['player_won']

# Align columns - crucial for ensuring test set has same features as train set
X_test = test_df[features]

# Initialize and train the model
print("Training a simple Logistic Regression model...")
model = LogisticRegression(random_state=42, max_iter=1000) # Remember to set random state!!
model.fit(X_train, y_train)
print("Model training complete.")

### 4. Creating the Submission File

The competition requires a `.csv` file with two columns: `battle_id` and `player_won`. Let's use our trained model to make predictions on the test set and format them correctly.

In [ ]:
# Make predictions on the test data
print("Generating predictions on the test set...")
test_predictions = model.predict(X_test)

# Use the provided sample file as the submission template.
sample_submission = pd.read_csv(sample_submission_path)
expected_columns = ['battle_id', 'player_won']
if list(sample_submission.columns) != expected_columns:
    raise ValueError(f"Expected submission columns {expected_columns}, got {list(sample_submission.columns)}")

# Match predictions to the template by battle_id, preserving its row order.
predictions_by_id = pd.Series(test_predictions, index=test_df['battle_id'])
submission_df = sample_submission.copy()
submission_df['player_won'] = submission_df['battle_id'].map(predictions_by_id)
if submission_df['player_won'].isna().any():
    raise ValueError("Some sample submission battle_id values have no matching test prediction.")
submission_df['player_won'] = submission_df['player_won'].astype(int)

submission_df.to_csv('submission.csv', index=False)

# Once you create the submission file, you will find it in /kaggle/working
print("\n'submission.csv' file created successfully!")
display(submission_df.head())

### 5. Submitting Your Results

Once you have generated your `submission.csv` file, there are two primary ways to submit it to the competition.

---

#### Method A: Submitting Directly from the Notebook (Recommended)

This is the standard method for code competitions. It ensures that your submission is linked to the code that produced it, which is crucial for reproducibility.

1.  **Save Your Work:** Click the **"Save Version"** button in the top-right corner of the notebook editor.
2.  **Run the Notebook:** In the pop-up window, select **"Save & Run All (Commit)"** and then click the **"Save"** button. This will run your entire notebook from top to bottom and save the output, including your `submission.csv` file.
3.  **Go to the Viewer:** Once the save process is complete, navigate to the notebook viewer page. 
4.  **Submit to Competition:** In the viewer, find the **"Submit to Competition"** section. This is usually located in the header of the output section or in the vertical "..." menu on the right side of the page. Clicking the **Submit** button this will submit your generated `submission.csv` file.

After submitting, you will see your score in the **"Submit to Competition"** section or in the [Public Leaderboard](https://www.kaggle.com/competitions/fds-pokemon-battles-prediction-2026/leaderboard?).

---

#### Method B: Manual Upload

You can also generate your predictions and submission file using any environment you prefer (this notebook, Google Colab, or your local machine).

1.  **Generate the `submission.csv` file** using your model.
2.  **Download the file** to your computer.
3.  **Navigate to the [Leaderboard Page](https://www.kaggle.com/competitions/fds-pokemon-battles-prediction-2026/leaderboard?)** and click on the **"Submit Predictions"** button.
4.  **Upload Your File:** Drag and drop or select your `submission.csv` file to upload it.

This method is quick, but keep in mind that for the final evaluation, you might be required to provide the code that generated your submission.

Good luck!